In [0]:
%sql
create connection if not exists earthquake
type HTTP
options(

  host =	'https://earthquake.usgs.gov',
  port=443,
  base_path = "/earthquakes/feed/v1.0/",
  bearer_token= "NA"
)

In [0]:
dbutils.widgets.text("catalog_name","dev")
catalog_name = dbutils.widgets.get("catalog_name")
print(catalog_name)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
conn = w.connections.get("earthquake")

base_url = f"{conn.options['host']}{conn.options['base_path']}"
print(base_url)

In [0]:
%sql
use catalog dev;
use schema bronze;
create volume if not exists earthquake_data;

In [0]:
import requests
import json
import datetime

url = f'{base_url}summary/all_day.geojson'
response = requests.get(url)
data = response.json()
if response.status_code != 200:
  raise Exception(f"Error: {response.status_code} - {response.text}")
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(f'/Volumes/dev/bronze/earthquake_data/earthquake_data_{current_date}.json',json.dumps(data),overwrite = True)